## Agent 智能体

### 快速入门

安装依赖

```
uv add langchain langchain-deepseek
```

配置环境, 在 `.env` 文件中添加 或者 系统环境变量
```
DEEPSEEK_API_KEY=your_api_key
```

In [22]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool

# 1.加载环境变量
load_dotenv() # 读取.env文件，加入系统环境变量中

# 2.定义工具，就是定义函数，给函数加装饰器 @tool
@tool
def get_weather(location: str):
    """查询指定location的天气"""
    return f"Current weather in {location} is sunny"

@tool
def square_root(x: float) -> float:
    """求x的平方根"""
    return x ** 0.5

# 3.定义Agent，绑定模型和工具，用函数create_agent
agent = create_agent(
    model="deepseek-v4-flash",
    tools=[get_weather, square_root]
)

# 4.调用Agent，用invoke函数，传入一个Dict，字典要有messages属性，里面是消息的列表
response = agent.invoke({
    "messages": [{"role": "user", "content": "杭州天气如何？100的平方根是多少？"}]
})

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

杭州天气如何？100的平方根是多少？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_Mvb6PLACnJqzLiSZnORA4929)
 Call ID: call_00_Mvb6PLACnJqzLiSZnORA4929
  Args:
    location: 杭州
  square_root (call_01_iH6lOUvilotMWczirZpA2274)
 Call ID: call_01_iH6lOUvilotMWczirZpA2274
  Args:
    x: 100
================================= Tool Message =================================
Name: get_weather

Current weather in 杭州 is sunny
================================= Tool Message =================================
Name: square_root

10.0
================================== Ai Message ==================================

杭州当前天气：晴。  
100 的平方根是：10（即 10.0）。


### 一、智能体（Agent）

#### 1.1 创建智能体

① 创建智能体，指定模型名，由Langchain初始化模型

In [23]:
from langchain.agents import create_agent
agent = create_agent(model="deepseek-v4-flash")

② 创建智能体，并使用创建好的model

`init_chat_model`的常见参数
- `model`: 模型名称
- `model_provider`: 模型提供商
- `api_key`: 模型的 API 密钥
- `base_url`: 模型提供商的 url 地址
- `max_tokens`: 最大生成 tokens 数
- `temperature`: 温度参数
- `extra_body`: 额外的请求参数

In [24]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}} # 指定是否开启思考模式 "enabled"、"disabled"
)
agent = create_agent(model=model)

#### 1.2 调用智能体

##### ① 阻塞式调用，使用 `invoke` 方法：

In [25]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "你是谁？"}]
})

print(response['messages'][-1].content)

你好！我是DeepSeek，由深度求索公司创造的AI助手。很高兴认识你！

我可以帮你解答问题、进行对话、处理文档、分析内容等等。我支持接收图片、PDF、Word、Excel等文件，并能从中提取文字信息进行分析。我的上下文容量高达1M，可以一次性处理像《三体》三部曲那么大体量的内容。

而且，我是完全免费的！无论是网页版还是App都可以免费使用，App还支持语音输入。如果需要联网搜索，记得手动点开联网搜索按键哦。

有什么我可以帮你的吗？😊


##### ② 流式调用，使用 `stream_events` 方法：

In [26]:
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "你是谁？"}]},
    version="v3"
)

for message in stream.messages:
    for text in message.text:
        print(text, end="", flush=True)

你好！我是DeepSeek，由深度求索公司创造的AI助手。😊

我可以帮你解答问题、处理文档、进行对话交流等。我支持：
- 📱 文件上传（图片、PDF、Word、Excel等，可以读取其中的文字信息，也能识别图片中的内容）
- 🔗 阅读链接内容
- 🌐 联网搜索（需要你手动开启）
- 💬 超长上下文（1M token，相当于能处理三体三部曲这样的长文本）

而且我是**完全免费**的！无论是网页版还是App都可以使用，App还支持语音输入功能。

有什么我可以帮你的吗？无论是学习、工作还是日常问题，尽管问我！✨

stream (`GraphRunStream` 类)
- `stream.messages`：模型消息六，每调用一次 LLM 就会对应一个独立的消息流
    - `message.text`：LLM 生成的最终文本内容
    - `message.tool_calls`：工具调用信息：工具调用的名称、参数等
    - `message.reasoning`：LLM 的推理内容
    - `message.output`：LLM 的输出的最终消息对象
- `stream.values`：执行过程中的 Agent 的 state 快照
- `stream.output`：最终的 Agent state
- `stream.subgraphs`：子Agent运行信息（多Agent嵌套时用）
- `stream.extensions`：自定义事件流
- `stream.tool_calls`：工具执行的生命周期，包括输入、输出增量、最终输出以及错误信息

#### 1.3 结构化输出

In [27]:
import json
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.messages import HumanMessage

#首先，我们定义一个类，用来封装模型要输出的数据：
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

# 然后，我们就可以创建智能体并设置结构化输出的格式了。
agent = create_agent(
    model='deepseek-chat',
    system_prompt="你是一个科幻作家，根据用户的要求创建一个太空之都。",
    response_format=CapitalInfo # 设置结构化输出的格式
)

response = agent.invoke(
    {"messages": [HumanMessage(content="月球的首都是什么?")]}
)

data_dict = response['structured_response'].model_dump()
pretty_json = json.dumps(data_dict, indent=4, ensure_ascii=False)
print(pretty_json)

{
    "name": "静海市",
    "location": "月球正面静海（Mare Tranquillitatis）西南缘，北纬约2°、东经约23°，紧邻阿波罗11号着陆点以西120公里；依托熔岩管穹顶与环形山坑壁建造，地表以上为多层透明穹顶群，地下为深达数百米的居住与工业层。",
    "vibe": "冷静、克制而繁盛。因为月面昼夜各十四天，城市以“永昼塔”与“永夜塔”两座人工光照中枢划分作息区，街头常年是恒定黄昏般的微光。居民称作“静海人”，说话轻缓、习惯低声，公共空间永远播放着低频率的环境白噪音以模拟地球的海潮声。地标是“回望广场”——一面巨大的地球全息投影，全天候悬挂在城市正上方，人们在那里举行婚礼、葬礼与条约签署。",
    "economy": "以氦-3开采与聚变燃料出口为支柱，配套月壤3D打印建材业、真空精密制造、低温超导材料，以及为地月航线服务的中转港与深空补给枢纽。城市同时是月球的行政、司法与金融中心，发行“静海元”，并通过“潮汐交易所”定价全月球的冰、氧与稀土配额。"
}


### 二、模型（Models）

####  2.1 直接调用模型

In [7]:
from langchain.chat_models import init_chat_model

# 初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}} # 指定是否开启思考模式 "enabled"、"disabled"
)

# 调用模型, 参数 str 或 list[dict]
response = model.invoke([
    {"role": "user", "content": "你叫什么名字？"}
])
print(response.pretty_print())

================================== Ai Message ==================================

你好！我叫DeepSeek，是由深度求索公司创造的AI助手。很高兴认识你！有什么我可以帮你的吗？😊
None


#### 2.2 模型结构化输出

In [9]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage, AIMessage

# 1.初始化模型 init_chat_model
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

# 2.定义 BaseModel 类，用来约束模型的输出，同时封装模型要输出的数据
from pydantic import BaseModel, Field

class CapitalInfo(BaseModel):
    name: str = Field(description="首都的名字")
    location: str = Field(description="The location of che capital")
    vibe: str = Field(description="The vibe of che capital")
    economy: str = Field(description="The economy source of che capital")

# 3.将BaseModel类与模型绑定，要求输出结构化结果
model_with_structured_output = model.with_structured_output(schema=CapitalInfo)

# 4.调用模型
response = model_with_structured_output.invoke([
    HumanMessage("你是科幻小说作家，帮我创造月球首都城市背景。"),
])

print(type(response)) # <class '__main__.CapitalInfo'>
print(response.model_dump_json(indent=2))

<class '__main__.CapitalInfo'>
{
  "name": "静海市",
  "location": "月球正面赤道附近的静海（Mare Tranquillitatis）火山玄武岩平原之下 40 至 220 米，主体沿一条废弃的熔岩管建成，地表仅露出一圈名为“月冕”的环形观景台与通讯塔阵。",
  "vibe": "冷静、克制而有秩序的“低重力优雅”。全域无天空，抬头是穹顶投映的人造星空与真实月面实景交叠；人们在六分之一重力下养成了滑行般的步态与极简的手势语。城市因真空而绝对安静，一切声音靠骨传导耳机传递，形成了“沉默的公共空间 + 喧闹的私人频道”这种独特社交礼仪。政治气质偏技术官僚与寡头议会混合，表面平和，暗流是地球移民与“月生代”之间的身份裂痕。",
  "economy": "三重支柱：①氦-3 开采与聚变燃料出口（月球经济的命脉，供应地月系电网）；②真空环境下的高纯度晶体、光纤与药物制造（“真空铸城”）；③低重力制造与航天器总装、以及面向地球富人的“月面居留权”与医疗康养产业。辅以水冰加工与月壤 3D 打印建材出口。"
}


### 三、消息（Messages）

所有消息都被统一封装成 `BaseMessage` 类

In [70]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain.agents import create_agent

# 定义工具并调用
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    print('tool调用：get_weather')
    return f'{location} 城市， 今天是晴天！'

# 创建Agent
agent = create_agent(model="deepseek-v4-flash", tools=[get_weather])

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        SystemMessage(content="你是智能问答助手"),
        HumanMessage(content="你好，我是虎哥"),
        AIMessage(content="你好，虎哥，很高兴认识你。"),
        HumanMessage(content="我的名字是什么？, 今天昆山天气如何？")
    ]
})

for message in response['messages']:
    message.pretty_print()

tool调用：get_weather
================================ System Message ================================

你是智能问答助手
================================ Human Message =================================

你好，我是虎哥
================================== Ai Message ==================================

你好，虎哥，很高兴认识你。
================================ Human Message =================================

我的名字是什么？, 今天昆山天气如何？
================================== Ai Message ==================================

虎哥，你刚才说你叫虎哥（虎哥是你的称呼）。我帮你查一下昆山的天气。
Tool Calls:
  get_weather (call_00_8Zzd1fh7NoamaBsqJvMu6390)
 Call ID: call_00_8Zzd1fh7NoamaBsqJvMu6390
  Args:
    location: 昆山
================================= Tool Message =================================
Name: get_weather

昆山 城市， 今天是晴天！
================================== Ai Message ==================================

虎哥，确认两件事：

1. **你的名字**：你自我介绍时说叫“虎哥”，所以我称你为虎哥。
2. **昆山天气**：今天是**晴天**。

需要我帮你做点别的吗，虎哥？


### 四、 提示词（Promts）

系统提示词（System Prompt）

In [19]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model = "deepseek-v4-flash",
    system_prompt="回答的时候要说好几次旺旺" # 设定角色、背景、任务等，每词调用都会跟用户问题一起拼接送给LLM
)

stream = agent.stream_events(
    {"messages": [HumanMessage(content="你是谁？")]},
    version="v3"
)

for message in stream.messages:
    for text in message.text:
        print(text, end="", flush=True)

我是你的AI助手，负责帮你答疑解惑、处理问题～旺旺、旺旺、旺旺！随时叫我旺旺助手就好。

提示词工程（Prompt Engineering）

In [20]:
# 提示词五要素
system_prompt = """
----------------提示词五要素-------------------
# 角色 Role：设定身份与专业视角、沟通风格和总体目标
# 任务 Task：明确要做什么，越具体越好
# 背景 Context：提供背景信息
# 约束 Constraints：不能做什么、要做到什么？比如字数、风格、禁用词、时间范围、是否允许编造等
# 输出格式 Format：规定输出格式，比如Markdown、JSON等

----------------Few-shot----------------
# 示例 Examples：输入输出样例
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-v4-flash",
    system_prompt=system_prompt
)

stream = agent.stream_events(
    {"messages": [HumanMessage(content="你是谁？")]},
    version="v3"
)

for message in stream.messages:
    for text in message.text:
        print(text, end="", flush=True)

我是 DeepSeek，由深度求索公司创造的 AI 助手。我可以帮你回答问题、处理信息、写作、翻译、编程等。有什么我可以帮你的？